# 💳 Credit Card Fraud Detection using Machine Learning

இந்த notebook-ல் நாம் பார்க்கப் போவது:
- ✅ Dataset load & explore
- ✅ Data preprocessing & balancing (SMOTE)
- ✅ Multiple ML models train பண்ணுவோம்
- ✅ Model evaluation (Confusion Matrix, ROC Curve)
- ✅ Best model save பண்ணுவோம்

**Dataset:** Kaggle Credit Card Fraud Detection (simulated version included)


## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install required libraries
!pip install imbalanced-learn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score,
    precision_score, recall_score, f1_score
)
from imblearn.over_sampling import SMOTE
import joblib

print('✅ All libraries imported successfully!')

## 📊 Step 2: Dataset உருவாக்கு (அல்லது Kaggle-இல் இருந்து load பண்ணு)

In [ ]:
# Option A: Synthetic dataset (Kaggle account இல்லாதவர்களுக்கு)
# Real Kaggle dataset வேண்டுமா? Option B கீழே பார்க்கவும்

from sklearn.datasets import make_classification

np.random.seed(42)

X, y = make_classification(
    n_samples=284807,
    n_features=29,
    n_informative=20,
    n_redundant=5,
    n_clusters_per_class=2,
    weights=[0.9983, 0.0017],  # Real fraud ratio: 0.17%
    flip_y=0,
    random_state=42
)

# DataFrame உருவாக்கு
feature_names = [f'V{i}' for i in range(1, 29)] + ['Amount']
df = pd.DataFrame(X, columns=feature_names)
df['Amount'] = np.abs(df['Amount']) * 100 + 10  # Realistic amount
df['Class'] = y
df['Time'] = np.random.uniform(0, 172792, size=len(df))

print('✅ Dataset ready!')
print(f'Total transactions: {len(df):,}')
print(f'Fraud cases: {df["Class"].sum():,} ({df["Class"].mean()*100:.2f}%)')
print(f'Legit cases: {(df["Class"]==0).sum():,}')

In [ ]:
# Option B: Kaggle-இல் இருந்து real dataset download (uncomment செய்யவும்)
# Kaggle account + API key தேவை

# !pip install kaggle -q
# from google.colab import files
# files.upload()  # kaggle.json upload பண்ணவும்
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d mlg-ulb/creditcardfraud --unzip
# df = pd.read_csv('creditcard.csv')
# print('✅ Real Kaggle dataset loaded!')

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Dataset overview
print('=== Dataset Shape ===')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')

print('\n=== First 5 rows ===')
df.head()

In [ ]:
# Missing values check
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else '✅ No missing values!')

print('\n=== Basic Statistics ===')
df[['Amount', 'Class']].describe()

In [ ]:
# Class distribution visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Credit Card Fraud Detection - EDA', fontsize=16, fontweight='bold')

# Plot 1: Class distribution
class_counts = df['Class'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Legitimate', 'Fraud'], class_counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 100, f'{v:,}\n({v/len(df)*100:.2f}%)', ha='center', fontsize=10)

# Plot 2: Transaction amount distribution
axes[1].hist(df[df['Class']==0]['Amount'], bins=50, alpha=0.7, color='#2ecc71', label='Legitimate')
axes[1].hist(df[df['Class']==1]['Amount'], bins=50, alpha=0.7, color='#e74c3c', label='Fraud')
axes[1].set_title('Transaction Amount Distribution', fontsize=13)
axes[1].set_xlabel('Amount (₹)')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].set_xlim(0, 500)

# Plot 3: Pie chart
axes[2].pie(class_counts.values, labels=['Legitimate', 'Fraud'],
            colors=colors, autopct='%1.3f%%', startangle=90,
            explode=(0, 0.1))
axes[2].set_title('Fraud vs Legitimate Split', fontsize=13)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 EDA plots saved!')

In [ ]:
# Correlation heatmap (top features)
plt.figure(figsize=(14, 10))
top_features = ['V1','V2','V3','V4','V5','V6','V7','V8','V9','V10',
                'V11','V12','V13','V14','Amount','Class']
corr_matrix = df[top_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdYlGn',
            center=0, linewidths=0.5, fmt='.2f')
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## ⚙️ Step 4: Data Preprocessing

In [ ]:
# Features & Target separate பண்ணு
X = df.drop(['Class', 'Time'], axis=1)
y = df['Class']

# Amount column scale பண்ணு
scaler = StandardScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])

# Train-Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set:     {X_test.shape[0]:,} samples')
print(f'\nTraining fraud cases: {y_train.sum():,} ({y_train.mean()*100:.2f}%)')

In [ ]:
# SMOTE - Class imbalance சரி செய்யவும்
print('⚡ SMOTE applying...')
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'\nBefore SMOTE:')
print(f'  Legit: {(y_train==0).sum():,} | Fraud: {(y_train==1).sum():,}')
print(f'\nAfter SMOTE:')
print(f'  Legit: {(y_train_sm==0).sum():,} | Fraud: {(y_train_sm==1).sum():,}')
print('✅ Dataset balanced!')

## 🤖 Step 5: ML Models Train பண்ணுவோம்

In [ ]:
# Models define பண்ணு
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}

print('🚀 Training models...\n')
for name, model in models.items():
    print(f'  Training {name}...', end=' ')
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'model':     model,
        'y_pred':    y_pred,
        'y_prob':    y_prob,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob)
    }
    print(f'✅ Done! (AUC: {results[name]["roc_auc"]:.4f})')

print('\n🎉 All models trained!')

## 📈 Step 6: Model Evaluation & Comparison

In [ ]:
# Results comparison table
metrics_df = pd.DataFrame({
    name: {
        'Accuracy':  f"{r['accuracy']*100:.2f}%",
        'Precision': f"{r['precision']*100:.2f}%",
        'Recall':    f"{r['recall']*100:.2f}%",
        'F1 Score':  f"{r['f1']*100:.2f}%",
        'ROC-AUC':   f"{r['roc_auc']:.4f}"
    }
    for name, r in results.items()
}).T

print('=== 📊 Model Performance Comparison ===')
metrics_df

In [ ]:
# Visualization: Metrics bar chart + ROC curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

# Plot 1: Bar chart comparison
model_names = list(results.keys())
short_names = ['LR', 'DT', 'RF', 'GB']
metric_vals = {
    'Precision': [results[m]['precision'] for m in model_names],
    'Recall':    [results[m]['recall'] for m in model_names],
    'F1 Score':  [results[m]['f1'] for m in model_names],
    'ROC-AUC':   [results[m]['roc_auc'] for m in model_names]
}

x = np.arange(len(short_names))
width = 0.2
colors_bar = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for i, (metric, values) in enumerate(metric_vals.items()):
    axes[0].bar(x + i * width, values, width, label=metric,
                color=colors_bar[i], alpha=0.85, edgecolor='black', linewidth=0.5)

axes[0].set_xlabel('Model')
axes[0].set_ylabel('Score')
axes[0].set_title('Metrics Comparison')
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(short_names)
axes[0].set_ylim(0, 1.1)
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: ROC curves
colors_roc = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
for (name, r), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    axes[1].plot(fpr, tpr, color=color, lw=2,
                 label=f'{name} (AUC={r["roc_auc"]:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves')
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrix - All models
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Confusion Matrices', fontsize=16, fontweight='bold')

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Fraud'],
                yticklabels=['Legit', 'Fraud'],
                linewidths=0.5)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 🏆 Step 7: Best Model & Feature Importance

In [ ]:
# Best model தேர்வு (ROC-AUC based)
best_name = max(results, key=lambda k: results[k]['roc_auc'])
best_model = results[best_name]['model']

print(f'🏆 Best Model: {best_name}')
print(f'   ROC-AUC:   {results[best_name]["roc_auc"]:.4f}')
print(f'   F1 Score:  {results[best_name]["f1"]:.4f}')
print(f'   Recall:    {results[best_name]["recall"]:.4f}')

print(f'\n📋 Detailed Classification Report:')
print(classification_report(y_test, results[best_name]['y_pred'],
                             target_names=['Legitimate', 'Fraud']))

In [ ]:
# Feature Importance (Tree-based models)
tree_models = ['Random Forest', 'Gradient Boosting', 'Decision Tree']
plot_model_name = next((m for m in tree_models if m in results), None)

if plot_model_name:
    importances = results[plot_model_name]['model'].feature_importances_
    feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)[:15]

    plt.figure(figsize=(10, 6))
    colors_fi = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(feat_imp)))
    feat_imp.plot(kind='barh', color=colors_fi[::-1], edgecolor='black', linewidth=0.5)
    plt.title(f'Top 15 Feature Importances ({plot_model_name})', fontsize=14, fontweight='bold')
    plt.xlabel('Importance Score')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nTop 5 important features:')
    for i, (feat, score) in enumerate(feat_imp.head().items(), 1):
        print(f'  {i}. {feat}: {score:.4f}')

## 💾 Step 8: Model Save & Real-time Prediction

In [ ]:
# Best model save
joblib.dump(best_model, 'fraud_detection_model.pkl')
joblib.dump(scaler, 'amount_scaler.pkl')
print(f'✅ Model saved: fraud_detection_model.pkl')
print(f'✅ Scaler saved: amount_scaler.pkl')

# Google Drive-ல் save பண்ண (optional)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp fraud_detection_model.pkl /content/drive/MyDrive/

In [ ]:
# Real-time prediction function
def predict_fraud(transaction_features, amount):
    """
    Predict if a transaction is fraudulent.
    transaction_features: list of V1-V28 values
    amount: transaction amount
    """
    model = joblib.load('fraud_detection_model.pkl')
    sc    = joblib.load('amount_scaler.pkl')

    # Feature vector உருவாக்கு
    scaled_amount = sc.transform([[amount]])[0][0]
    features = transaction_features + [scaled_amount]
    X_input = np.array(features).reshape(1, -1)

    prediction = model.predict(X_input)[0]
    probability = model.predict_proba(X_input)[0][1]

    result = '🚨 FRAUD' if prediction == 1 else '✅ LEGITIMATE'
    print(f'Transaction Amount: ₹{amount:.2f}')
    print(f'Prediction: {result}')
    print(f'Fraud Probability: {probability*100:.2f}%')
    print(f'Risk Level: {"HIGH" if probability > 0.7 else "MEDIUM" if probability > 0.3 else "LOW"}')
    return prediction, probability

# Test prediction
print('=== 🧪 Sample Prediction Test ===')
sample = X_test.iloc[0].drop('Amount', errors='ignore').tolist()[:28]
sample_amount = 150.0
pred, prob = predict_fraud(sample, sample_amount)
print(f'\nActual label: {"FRAUD" if y_test.iloc[0]==1 else "LEGITIMATE"}')

## 📋 Step 9: Final Summary

In [ ]:
print('='*55)
print('  💳 CREDIT CARD FRAUD DETECTION - FINAL SUMMARY')
print('='*55)
print(f'  Dataset size    : {len(df):,} transactions')
print(f'  Fraud rate      : {df["Class"].mean()*100:.2f}%')
print(f'  Features used   : {X.shape[1]}')
print(f'  Balancing method: SMOTE')
print(f'  Models trained  : {len(models)}')
print()
print('  Model Rankings (by ROC-AUC):')
ranked = sorted(results.items(), key=lambda x: x[1]['roc_auc'], reverse=True)
medals = ['🥇', '🥈', '🥉', '  4.']
for i, (name, r) in enumerate(ranked):
    print(f'  {medals[i]} {name:<22} AUC={r["roc_auc"]:.4f}  F1={r["f1"]:.4f}')
print()
print(f'  🏆 Winner       : {best_name}')
print(f'  📁 Saved as     : fraud_detection_model.pkl')
print('='*55)
print('  ✅ Project Complete! Happy Coding! 🎉')
print('='*55)